In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error
import matplotlib.pyplot as plt
import json
import os
import gspread
from google.oauth2.service_account import Credentials
from gspread_dataframe import get_as_dataframe
import openai

# ===== 1️⃣ 讀取環境變數中的 OpenAI API Key =====
openai_api_key = os.getenv("OPENAI_API_KEY")  # 讀取 GitHub Secrets
client = openai.OpenAI(api_key=openai_api_key)

# ===== 2️⃣ Google Sheets 授權 (使用 GitHub Secrets) =====
service_account_info = json.loads(os.getenv("GCP_SERVICE_ACCOUNT"))  # 讀取 GitHub Secrets
creds = Credentials.from_service_account_info(service_account_info, scopes=["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"])
gc = gspread.authorize(creds)

# ===== 3️⃣ 讀取 Google Sheet 資料 =====
SHEET_URL = "https://docs.google.com/spreadsheets/d/1HlFKyX0rtl5y3_OU31KnMub75MZaMfnU5hG98LNNhJk/edit?usp=sharing"  # 請替換成你的 Google Sheet 連結
sh = gc.open_by_url(SHEET_URL)
worksheet = sh.sheet1  # 讀取第一個工作表
df = get_as_dataframe(worksheet, evaluate_formulas=True)

# ===== 4️⃣ 資料前處理 =====
df = df.dropna()
df = df.rename(columns={
    "成交金額 (億)": "Spot Market Trading Volume",
    "外資及陸資(不含外資自營商)": "Foreign Investors Net Buy/Sell",
    "大台指期未平倉口數淨額": "Net Open Interest of Large TAIEX Futures",
    "小台指期未平倉口數淨額": "Net Open Interest of Mini TAIEX Futures",
    "加權指數": "TAIEX"
})

df['漲跌'] = (df['TAIEX'].diff() > 0).astype(int)  # 計算分類標籤
df = df.dropna()

X = df[["Spot Market Trading Volume", "Foreign Investors Net Buy/Sell", "Net Open Interest of Large TAIEX Futures", "Net Open Interest of Mini TAIEX Futures"]]
y = df["漲跌"]

# ===== 5️⃣ 分割訓練集與測試集 =====
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ===== 6️⃣ 使用 Grid Search 進行 XGBClassifier 超參數調優 =====
param_grid_class = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search_class = GridSearchCV(estimator=xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
                                  param_grid=param_grid_class,
                                  scoring='accuracy',
                                  cv=5,
                                  verbose=1,
                                  n_jobs=-1)

grid_search_class.fit(X_train, y_train)

# 儲存分類模型結果
best_params_class = grid_search_class.best_params_
best_class_model = xgb.XGBClassifier(**best_params_class, use_label_encoder=False, eval_metric='logloss')
best_class_model.fit(X_train, y_train)

# ===== 7️⃣ 預測並評估最佳分類模型 =====
y_pred_best = best_class_model.predict(X_test)
best_accuracy = accuracy_score(y_test, y_pred_best)
class_report = classification_report(y_test, y_pred_best)

# ===== 8️⃣ 使用 Grid Search 進行 XGBRegressor 超參數調優 =====
y_reg = df["TAIEX"]  # 目標變數為加權指數的連續數值
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

param_grid_reg = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search_reg = GridSearchCV(estimator=xgb.XGBRegressor(),
                                param_grid=param_grid_reg,
                                scoring='neg_mean_squared_error',
                                cv=5,
                                verbose=1,
                                n_jobs=-1)

grid_search_reg.fit(X_train_reg, y_train_reg)

# 儲存迴歸模型結果
best_params_reg = grid_search_reg.best_params_
best_reg_model = xgb.XGBRegressor(**best_params_reg)
best_reg_model.fit(X_train_reg, y_train_reg)

y_reg_pred_best = best_reg_model.predict(X_test_reg)
reg_mse_best = mean_squared_error(y_test_reg, y_reg_pred_best)

# ===== 9️⃣ 統一輸出結果 =====
results = {
    "Best XGBClassifier Parameters": best_params_class,
    "Best XGBClassifier Accuracy (Test Set)": best_accuracy,
    "Classification Report": class_report,
    "Best XGBRegressor Parameters": best_params_reg,
    "Best XGBRegressor MSE (Test Set)": reg_mse_best
}

json_data = json.dumps(results, indent=4)

# ===== 🔟 取得 OpenAI API 的投資建議 =====
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "你是一名專業的投資顧問，負責根據市場數據提供投資建議"},
        {"role": "user", "content": f"請幫我分析這些模型結果，並給近期投資建議（字數不超過100字），每個段落請換下一行：\n{json_data}"}
    ]
)

report_text = response.choices[0].message.content
print("📊 投資建議：\n")
print(report_text)

# ===== 🔟 將 ChatGPT 報告寫入 Google Sheets =====
worksheet.append_row([report_text])  # 每次新增一行
print("✅ ChatGPT 報告已成功寫入 Google Sheets")
